In [29]:
import os 
import sys 
import ipynbname
import fnmatch
from pathlib import Path

import logging
from dotenv import load_dotenv
load_dotenv()

python_dir=os.getenv('python_dir')

if python_dir not in sys.path: 
    print(f"adding {python_dir} to sys path")
    sys.path.append(python_dir)

from common_utils import setup_logger
log_dir=os.getenv('log_dir')
log_file=os.path.join(log_dir, f"{ipynbname.name()}.log")
setup_logger('INFO',log_file)

logging.info(f"start  {ipynbname.name()}")

In [30]:
from pathlib import Path 
tmp_dir=os.getenv('tmp_dir')
workspace_dir=os.path.join(tmp_dir, "aimpro",f"{os.getpid()}")
Path(workspace_dir).mkdir(parents=True,exist_ok=True)
workspace_dir

'/tmp/aimpro/6665'

In [ ]:
video_file_ext="MP4"
extract_fps=3
extract_img_fmt="jpg"
detection_chunk_size=60

In [44]:
raw_video_dir=r"/mnt/c/data/temp/vid_ml_files/20260529-vs-elthem"
left_video_dir=os.path.join(raw_video_dir, 'left')
right_video_dir=os.path.join(raw_video_dir, 'right')

def gen_concat_cmd(video_dir, concat_file_name): 
    files=fnmatch.filter(os.listdir(video_dir), f"*.{video_file_ext}")
    files=[os.path.join(video_dir, f) for f in files ]
    sorted(files, key=os.path.getctime)
    list_file_name=os.path.join(workspace_dir, f"{concat_file_name}.list.txt")
    with open(list_file_name, 'w') as h : 
        for f in files:
            h.write(f"file '{f}' \n")
    txt=  f"ffmpeg -f concat -safe 0 -i {list_file_name} -c copy {os.path.join(workspace_dir, concat_file_name)}"
    pyperclip.copy(txt)
    return txt 


In [42]:
gen_concat_cmd(left_video_dir, 'left_all_in_one.MP4')

'ffmpeg -f concat -safe 0 -i /tmp/aimpro/6665/left_all_in_one.MP4.list.txt -c copy /tmp/aimpro/6665/left_all_in_one.MP4'

In [46]:
gen_concat_cmd(right_video_dir, 'right_all_in_one.MP4')

'ffmpeg -f concat -safe 0 -i /tmp/aimpro/6665/right_all_in_one.MP4.list.txt -c copy /tmp/aimpro/6665/right_all_in_one.MP4'

In [47]:

# left_video_file=os.path.join(raw_video_dir, 'left','20260516181012_000004.MP4')
# right_video_file=os.path.join(raw_video_dir, 'right','r_20260516181403_000006.MP4')
left_video_file=os.path.join(workspace_dir, 'left_all_in_one.MP4')
right_video_file=os.path.join(workspace_dir, 'right_all_in_one.MP4')

left_video_file, right_video_file

('/tmp/aimpro/6665/left_all_in_one.MP4',
 '/tmp/aimpro/6665/right_all_in_one.MP4')

In [48]:
from video.synchronizer import * 
raw_offset, meaning = find_offset_seconds(workspace_dir,left_video_file, right_video_file)
offset=round(raw_offset)
raw_offset, offset, meaning

(-3.39775, -3, 'video B started 3.39775 earlier than video A.')

In [49]:
from video.ffmpeg_wrapper import * 
left_video_duration= get_video_duration(left_video_file)
right_video_duration= get_video_duration(right_video_file)
left_video_duration,right_video_duration 


(3187.467, 3189.084)

In [50]:
left_frames_dir=os.path.join(workspace_dir, 'left')
Path(left_frames_dir).mkdir(parents=True, exist_ok=True)

right_frames_dir=os.path.join(workspace_dir, 'right')
Path(right_frames_dir).mkdir(parents=True, exist_ok=True)

In [51]:
def get_start_end_time(offset, left_duration, right_duration): 
    if offset <0: 
        # right start early
        right_start=-1*offset
        new_right_duration =  right_duration+offset 
        min_duration =min(left_duration, new_right_duration)
        return 0, min_duration, right_start, right_start+min_duration

    elif offset>0: 
        left_start=offset 
        new_left_duration = left_duration-offset
        min_duration=min(new_left_duration, right_duration)
        return left_start, left_start+ min_duration, 0, min_duration
ts= get_start_end_time(offset, left_video_duration, right_video_duration)
ts =[format_seconds_to_hhmmss(t) for t in ts]
left_start, left_end, right_start, right_end= ts[0],ts[1],ts[2],ts[3]
left_start, left_end, right_start, right_end

('00:00:00', '00:53:06.083', '00:00:03', '00:53:09.083')

In [52]:
file_name_pattern=f"%d.{extract_img_fmt}"
output = os.path.join(left_frames_dir, file_name_pattern)
cmd=f"ffmpeg -ss {left_start} -to {left_end} -i {left_video_file} -vf scale=1280:720,fps={extract_fps} {output}"
stdout, stderr = call_command_line(cmd)
print(stdout)


In [53]:
output = os.path.join(right_frames_dir, file_name_pattern)
cmd=f"ffmpeg -ss {right_start} -to {right_end} -i {right_video_file} -vf scale=1280:720,fps={extract_fps} {output}"
stdout, stderr = call_command_line(cmd)
print(stdout)


In [54]:
def rename_img_file_to_seconds_index(fd):
    files = fnmatch.filter(os.listdir(fd), f"*.{extract_img_fmt}")
    for f in files: 
        parts=f.split(".")
        file_index, ext=int(parts[0])-1, parts[1]
        second, index_in_second=divmod(file_index,extract_fps)
        new_file_name=f"{second}-{index_in_second}.{ext}"
        shutil.move(os.path.join(fd,f), os.path.join(fd,new_file_name))
rename_img_file_to_seconds_index(left_frames_dir)
rename_img_file_to_seconds_index(right_frames_dir)

In [ ]:
from ml.detectors import * 
import pandas  as pd 

def detect_basketball_and_players(video_frame_dir): 
    files=fnmatch.filter(os.listdir(video_frame_dir), f"*.{extract_img_fmt}")
    logging.info(files)
    df=pd.DataFrame(data=files, columns=["file_name"])
    df['chunk']=range(df.shape[0])
    df['chunk']=df.chunk//detection_chunk_size
    def detect_chunk(gdf):
        logging.info(f"object detection chuck : {gdf.name}")
        files=[os.path.join(video_frame_dir,f) for f in gdf.file_name] 
        results=detect_objects_on_multiple_images(files)
        trios=[analyze_result(r) for r in results]
        num_of_basketballs, num_of_players , num_of_possessions= zip (*trios)
        gdf['has_baseketball']=num_of_basketballs
        gdf['has_baseketball']=gdf.has_baseketball>0
        gdf['has_possession']=num_of_possessions
        gdf['has_possession']=gdf.has_possession>0
        gdf['total_num_of_players']=num_of_players
        return gdf 
    ret= df.groupby('chunk').apply(detect_chunk)
    ret.sort_values(by="file_name", inplace=True)
    ret.to_csv(os.path.join(video_frame_dir, "frame_information.csv"), index=False)
    # return pd.concat(dfs)
    return ret
detect_basketball_and_players(left_frames_dir).shape , detect_basketball_and_players(right_frames_dir).shape 


In [ ]:
def agg_frame_information_df(df):
    def extract_second_and_index(fn):
        parts = fn.split(".")[0].split("-") 
        return parts[0], parts[1]
    pairs=df.file_name.apply(extract_second_and_index)
    df['second'], df["index_within_second"]=zip (*pairs)
    df['second'], df["index_within_second"]=df.second.astype(int), df.index_within_second.astype(int)
    adf=df.groupby('second').agg(
        {
            "has_baseketball":'any',
            "total_num_of_players":'sum' ,
            "has_possession":'any'   
        } 
    ).reset_index()
    adf.sort_values(by='second', inplace=True)
    return adf 

    

left_frames_df=pd.read_csv(os.path.join(left_frames_dir, "frame_information.csv"), index_col=None)
left_frames_df=agg_frame_information_df(left_frames_df)

right_frames_df=pd.read_csv(os.path.join(right_frames_dir, "frame_information.csv"), index_col=None)
right_frames_df=agg_frame_information_df(right_frames_df)

left_frames_df.shape, right_frames_df.shape 


((598, 4), (598, 4))

In [23]:
left_frames_df.head(2)

,second,has_baseketball,total_num_of_players,has_possession
0,0,False,6,False
1,1,False,6,False


In [24]:
mdf=left_frames_df.merge(right_frames_df, how='inner', on='second', suffixes=("_l","_r"))
mdf.head(2) 

,second,has_baseketball_l,total_num_of_players_l,has_possession_l,has_baseketball_r,total_num_of_players_r,has_possession_r
0,0,False,6,False,False,28,True
1,1,False,6,False,False,27,True


In [25]:
def smooth_left_right(ac): 
    s=ac.copy()
    prev,nxt=s.shift(1),s.shift(-1)
    mask=(s!=prev) & (s!=nxt) & (prev==nxt)
    ret=s.copy()
    ret[mask]=prev[mask]
    return ret 

def decide_active_camera(r):
    ball_on_left= r["has_possession_l"]
    ball_on_right= r["has_possession_r"]

    only_on_left=ball_on_left and not ball_on_right
    only_on_right=ball_on_right and not ball_on_left

    if only_on_left:
         return 'left'
    elif only_on_right: 
        return 'right'
    else: 
        if r['total_num_of_players_l']>r['total_num_of_players_r']:
                return 'left'
        elif r['total_num_of_players_l']<r['total_num_of_players_r']:
                return 'right'
    return None
mdf['active_camera']=mdf.apply(decide_active_camera, axis=1)
mdf.ffill(inplace=True)
mdf.sort_values(by='second', ascending=True, inplace=True)
#double smooth 
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf.to_csv(os.path.join(workspace_dir,'active_camera.csv'), index=False)
mdf.head(2)

,second,has_baseketball_l,total_num_of_players_l,has_possession_l,has_baseketball_r,total_num_of_players_r,has_possession_r,active_camera
0,0,False,6,False,False,28,True,right
1,1,False,6,False,False,27,True,right


In [26]:
def gen_hhmmss(side, offset, second): 
    #if offset <0 , then right start early; otherwise left start early. 
    if offset<0: 
        if side =='right': 
            second =float(second) +offset*(-1)
    else: 
        if side=='left': 
            second =float(second) +offset
    return format_seconds_to_hhmmss(second) 

def gen_ffmpeg_extract_commands(mdf):
    def gen_cmd(start_r,last_r): 
        if start_r.active_camera=='left':
            start_hhmmss=gen_hhmmss("left", offset, start_r.second)
            to_hhmmss=gen_hhmmss("left", offset, last_r.second)
            video_file_name=left_video_file
        else:
            start_hhmmss=gen_hhmmss("right", offset, start_r.second)
            to_hhmmss=gen_hhmmss("right", offset, last_r.second)
            video_file_name=right_video_file
        return [f"ffmpeg -ss {start_hhmmss} -to {to_hhmmss} -i {video_file_name}  -c copy part_to_be_replaced"]
    
    start_r, last_r=None, None  
    cmds=[]
    for r in mdf.itertuples():
        if start_r is None: 
            start_r, last_r=r, r 
            continue
        else: 
            if r.active_camera!=last_r.active_camera: 
                #need swtich. 
                cmds.extend(gen_cmd(start_r, last_r))
                start_r, last_r=r, r 
            else:
                last_r=r 

    if start_r and last_r: 
        cmds.extend(gen_cmd(start_r,last_r))       
    return cmds 
extract_cmds = gen_ffmpeg_extract_commands(mdf)
extract_cmds[-1]


'ffmpeg -ss 00:09:54 -to 00:09:57 -i /mnt/c/data/temp/vid_ml_files/war_vs_bul_domestic/left/20260516175012_000002.MP4  -c copy part_to_be_replaced'

In [27]:
import pyperclip
def post_process_extract_cmds(extract_cmds): 
    def replace(index, cmd): 
        part=os.path.join(workspace_dir, f"part-{index}.MP4")
        return cmd.replace('part_to_be_replaced', part)
    extract_cmds= [replace(i,c) for i, c in enumerate(extract_cmds) ]
    text="\n".join(extract_cmds)
    pyperclip.copy(text)

    #write to sh file. 
    with open(os.path.join(workspace_dir,'extract_part.sh'), 'w') as h: 
        h.write(text)
    
    return extract_cmds
    
extract_cmds=post_process_extract_cmds(extract_cmds)
# for c in extract_cmds: 
#     o,e = call_command_line(extract_cmds)
#     if o: 
#         print(o)
#     if e: 
#         print(e)

In [ ]:
! bash {workspace_dir}/extract_part.sh 

In [ ]:
def gen_concat_command(extract_cmds, output_file_name_path): 
    list_file_name=os.path.join(workspace_dir, 'part-list.txt')
    with open(list_file_name, 'w') as h: 
        for i in range(len(extract_cmds)):
            p=os.path.join(workspace_dir,f"part-{i}.MP4")  
            h.write(f"file '{p}' \n")
    txt=  f"ffmpeg -f concat -safe 0 -i {list_file_name} -c copy {output_file_name_path}"
    pyperclip.copy(txt)
    return txt 
    
concat_cmd= gen_concat_command(extract_cmds, os.path.join(raw_video_dir,'20260516-war1-bul3-domestic-q4.MP4'))
concat_cmd

'ffmpeg -f concat -safe 0 -i /tmp/aimpro/6665/part-list.txt -c copy /mnt/c/data/temp/vid_ml_files/war_vs_bul_domestic/output.MP4'

In [ ]:
#! rm -rf {workspace_dir}